In [1]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # or paste token string directly


import requests
import json
import time
import yaml
import os
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
DRIVE_BASE = '/content/drive/MyDrive/iam_rag_project/'

# ── Target orgs and topics ────────────────────────────────────────────────────
# aws-samples: short focused examples, ideal for task-specific policies
# awslabs: more production-ready, good for complex patterns
TARGET_ORGS = ["aws-samples", "awslabs"]

# Topics that signal serverless/container repos with IAM policies
TARGET_TOPICS = [
    "serverless", "lambda", "ecs", "fargate",
    "api-gateway", "dynamodb", "sqs", "eventbridge"
]

Mounted at /content/drive


In [2]:
!pip install -q voyageai rank_bm25 transformers torch tqdm faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 30.1 MB/s eta 0:00:00


In [ ]:
#scrape IAM policies from official AWS repos
# ── Step 1: Collect candidate repos ──────────────────────────────────────────

def get_repos_by_topic(org, topic, max_repos=30):
    """Get repos from an org filtered by topic."""
    url = "https://api.github.com/search/repositories"
    params = {
        "q": f"org:{org} topic:{topic}",
        "sort": "stars",
        "per_page": max_repos
    }
    r = requests.get(url, headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"  Error {r.status_code} for {org}/{topic}")
        return []
    return r.json().get("items", [])


print("Collecting candidate repos...")
seen_repo_ids = set()
candidate_repos = []

for org in TARGET_ORGS:
    for topic in TARGET_TOPICS:
        repos = get_repos_by_topic(org, topic, max_repos=20)
        for repo in repos:
            if repo["id"] not in seen_repo_ids:
                seen_repo_ids.add(repo["id"])
                candidate_repos.append({
                    "owner": repo["owner"]["login"],
                    "name":  repo["name"],
                    "full_name": repo["full_name"],
                    "stars": repo["stargazers_count"],
                    "url":   repo["html_url"]
                })
        time.sleep(0.5)  # respect rate limit

# Sort by stars — higher starred repos tend to be better quality
candidate_repos.sort(key=lambda x: x["stars"], reverse=True)
print(f"Found {len(candidate_repos)} unique candidate repos")
print(f"Top 10 by stars:")
for r in candidate_repos[:10]:
    print(f"  ★{r['stars']:<6} {r['full_name']}")

Found 147 unique candidate repos
Top 10 by stars:
  ★7571   awslabs/agent-squad
  ★4265   aws-samples/aws-serverless-workshops
  ★2325   aws-samples/aws-serverless-airline-booking
  ★1788   aws-samples/serverless-patterns
  ★1594   aws-samples/lambda-refarch-webapp
  ★1559   aws-samples/custom-lens-wa-hub
  ★1316   aws-samples/generative-ai-use-cases
  ★1285   aws-samples/bedrock-chat
  ★1148   aws-samples/aws-serverless-ecommerce-platform
  ★892    awslabs/fargatecli


In [ ]:
# ── Step 2: Find CloudFormation template files in each repo ──────────────────

def find_cfn_files_in_repo(owner, repo_name, max_files=10):
    """
    Find CloudFormation templates by browsing the repo tree directly.
    Uses /git/trees API instead of code search — much higher rate limit.
    """
    # Get the default branch first
    r = requests.get(
        f"https://api.github.com/repos/{owner}/{repo_name}",
        headers=HEADERS, timeout=10
    )
    if r.status_code != 200:
        return []
    default_branch = r.json().get("default_branch", "main")
    time.sleep(0.3)

    # Get full file tree recursively
    r = requests.get(
        f"https://api.github.com/repos/{owner}/{repo_name}/git/trees/"
        f"{default_branch}?recursive=1",
        headers=HEADERS, timeout=15
    )
    if r.status_code != 200:
        return []
    time.sleep(0.3)

    tree = r.json().get("tree", [])

    # Find files that look like CloudFormation templates by name/extension
    cfn_files = []
    for item in tree:
        if item.get("type") != "blob":
            continue
        path = item.get("path", "")
        path_lower = path.lower()

        if not (path_lower.endswith(".yaml")
                or path_lower.endswith(".yml")
                or path_lower.endswith(".json")):
            continue

        skip_dirs = [
            "node_modules", ".github", "test/",
            "tests/", "__pycache__", "dist/", "build/",
            "docs/", "examples/", ".aws-sam/"
        ]
        if any(s in path_lower for s in skip_dirs):
            continue

        # Accept any yaml/json — content check happens in extract_policies_from_cfn
        cfn_files.append({
            "path":     path,
            "html_url": (
                f"https://github.com/{owner}/{repo_name}"
                f"/blob/{default_branch}/{path}"
            )
        })

        if len(cfn_files) >= max_files:
            break

    return cfn_files, default_branch


def get_file_content(download_url):
    """Fetch raw file content."""
    r = requests.get(download_url, headers=HEADERS, timeout=10)
    if r.status_code == 200:
        return r.text
    return None

In [ ]:
# ── Step 3: Extract IAM policies from CloudFormation templates ───────────────

def extract_policies_from_cfn(content, source_url):
    template = None
    try:
        template = json.loads(content)
    except json.JSONDecodeError:
        try:
            template = yaml.load(content, Loader=SafeLineLoader)
        except Exception:
            return []

    if not isinstance(template, dict):
        return []

    if "Resources" not in template and "AWSTemplateFormatVersion" not in template:
        return []

    resources = template.get("Resources", {})
    if not isinstance(resources, dict):
        return []

    policies = []

    for resource_name, resource in resources.items():
        if not isinstance(resource, dict):
            continue
        rtype = resource.get("Type", "")
        props = resource.get("Properties", {})
        if not isinstance(props, dict):
            continue

        if rtype == "AWS::IAM::Policy":
            doc = props.get("PolicyDocument", {})
            if isinstance(doc, dict) and "Statement" in doc:
                policies.append({
                    "resource_name":  resource_name,
                    "resource_type":  rtype,
                    "policy_document": doc,
                    "source_url":     source_url
                })

        elif rtype == "AWS::IAM::Role":
            all_role_actions = []
            for inline in props.get("Policies", []):
                if not isinstance(inline, dict):
                    continue
                doc = inline.get("PolicyDocument", {})
                if isinstance(doc, dict) and "Statement" in doc:
                    all_role_actions.extend(
                        extract_actions_from_policy(doc)
                    )
            if all_role_actions:
                merged_doc = {
                    "Version": "2012-10-17",
                    "Statement": [{
                        "Effect":   "Allow",
                        "Action":   list(set(all_role_actions)),
                        "Resource": "*"
                    }]
                }
                policies.append({
                    "resource_name":   resource_name,
                    "resource_type":   "AWS::IAM::Role/MergedInline",
                    "policy_document": merged_doc,
                    "source_url":      source_url
                })

        elif rtype == "AWS::IAM::ManagedPolicy":
            doc = props.get("PolicyDocument", {})
            if isinstance(doc, dict) and "Statement" in doc:
                policies.append({
                    "resource_name":   resource_name,
                    "resource_type":   rtype,
                    "policy_document": doc,
                    "source_url":      source_url
                })

    return policies

In [ ]:
# ── Step 4: Quality filter ────────────────────────────────────────────────────

def extract_actions_from_policy(policy_doc):
    """Flatten all Allow actions from a policy document."""
    actions = []
    for stmt in policy_doc.get("Statement", []):
        if not isinstance(stmt, dict):
            continue
        if stmt.get("Effect") != "Allow":
            continue
        raw = stmt.get("Action", [])
        if isinstance(raw, str):
            raw = [raw]
        actions.extend(raw)
    return actions


def is_useful_policy(policy_doc, min_actions=2, max_actions=15):
    """
    Filter for policies useful as ground truth evaluation data.
    Rejects overly broad, overly narrow, or wildcard-heavy policies.
    """
    actions = extract_actions_from_policy(policy_doc)

    if not actions:
        return False

    # Reject admin/full-access policies
    if "*" in actions:
        return False
    broad_wildcards = [a for a in actions if a.endswith(":*")]
    if len(broad_wildcards) > 0:
        return False

    # Reject CloudFormation intrinsic functions that weren't resolved
    # (sometimes YAML templates have Ref/Fn::Sub in action fields)
    for a in actions:
        if not isinstance(a, str) or ":" not in a:
            return False

    # Enforce action count bounds
    if len(actions) < min_actions:
        return False
    if len(actions) > max_actions:
        return False

    # Must touch at least 1 of our target services
    target_prefixes = {
        "s3", "lambda", "dynamodb", "sqs", "sns", "logs",
        "cloudwatch", "events", "ecs", "ecr", "ssm", "kms",
        "secretsmanager", "sts", "codedeploy", "apigateway",
        "ec2", "iam", "xray", "cloudformation"
    }
    prefixes = {a.split(":")[0].lower() for a in actions}
    if not prefixes & target_prefixes:
        return False

    return True

In [ ]:
# ── Step 5: Run the full pipeline ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("SCRAPING IAM POLICIES FROM AWS GITHUB REPOS")
print("=" * 60)

all_policies   = []
repos_searched = 0
files_fetched  = 0
TARGET_COUNT   = 80  # aim for this policy count

for repo in tqdm(candidate_repos):
    if len(all_policies) >= TARGET_COUNT:
        break

    owner     = repo["owner"]
    repo_name = repo["name"]

    # Find CloudFormation files in this repo using tree API (no rate limit issues)
    cfn_files, default_branch = find_cfn_files_in_repo(
        owner, repo_name, max_files=5
    )
    repos_searched += 1

    if not cfn_files:
        continue

    for file_item in cfn_files:
        if len(all_policies) >= TARGET_COUNT:
            break

        raw_url = (
            f"https://raw.githubusercontent.com/"
            f"{owner}/{repo_name}/{default_branch}/{file_item['path']}"
        )

        content = get_file_content(raw_url)
        if not content:
            continue

        files_fetched += 1

        # Extract and filter policies
        extracted = extract_policies_from_cfn(content, file_item["html_url"])
        for policy in extracted:
            if is_useful_policy(policy["policy_document"]):
                policy["repo"]   = repo["full_name"]
                policy["stars"]  = repo["stars"]
                policy["actions"] = extract_actions_from_policy(
                    policy["policy_document"]
                )
                all_policies.append(policy)

        time.sleep(0.3)

print(f"\nRepos searched:    {repos_searched}")
print(f"Files fetched:     {files_fetched}")
print(f"Policies collected: {len(all_policies)}")


SCRAPING IAM POLICIES FROM AWS GITHUB REPOS


100%|██████████| 147/147 [09:03<00:00,  3.70s/it]


Repos searched:    147
Files fetched:     553
Policies collected: 5


In [ ]:
# ── Step 6: Deduplicate by action set ────────────────────────────────────────

seen_action_sets = set()
unique_policies  = []

for p in all_policies:
    key = frozenset(p["actions"])
    if key not in seen_action_sets:
        seen_action_sets.add(key)
        unique_policies.append(p)

print(f"After deduplication: {len(unique_policies)} unique policies")

After deduplication: 50 unique policies


In [ ]:
# ── Step 7: Deduplicate and save ──────────────────────────────────────────────
import datetime

class SafeEncoder(json.JSONEncoder):
    """Handles types that standard json.dump can't serialize."""
    def default(self, obj):
        if isinstance(obj, (datetime.date, datetime.datetime)):
            return obj.isoformat()
        if isinstance(obj, set):
            return list(obj)
        return str(obj)  # fallback — convert anything else to string

seen_action_sets = set()
unique_policies  = []

for p in all_policies:
    key = frozenset(p["actions"])
    if key not in seen_action_sets:
        seen_action_sets.add(key)
        unique_policies.append(p)

print(f"After deduplication: {len(unique_policies)} unique policies")

output_path = os.path.join(
    DRIVE_BASE, "data/processed/github_iam_policies.json"
)
with open(output_path, "w") as f:
    json.dump(unique_policies, f, indent=2, cls=SafeEncoder)

print(f"\n✓ Saved {len(unique_policies)} policies to:")
print(f"  {output_path}")

After deduplication: 50 unique policies

✓ Saved 50 policies to:
  /content/drive/MyDrive/iam_rag_project/data/processed/github_iam_policies.json


In [ ]:
import json
import os

# Load the scraped policies
input_path = os.path.join(DRIVE_BASE, "data/processed/github_iam_policies.json")
with open(input_path) as f:
    policies = json.load(f)

print(f"Starting count: {len(policies)} policies")

# ── Step 1: Deduplicate actions within each policy ────────────────────────────
for p in policies:
    p["actions"] = sorted(list(set(p["actions"])))
    # Update policy_document to match deduplicated actions
    p["policy_document"]["Statement"][0]["Action"] = p["actions"]

print("✓ Deduplicated actions within each policy")

# ── Step 2: Drop too-minimal policies ────────────────────────────────────────
# Policies that are only CloudWatch logs actions are not meaningful tasks —
# every Lambda has these and they don't represent a queryable scenario
def is_only_logs(actions):
    return all(a.startswith("logs:") for a in actions)

before = len(policies)
policies = [p for p in policies if not is_only_logs(p["actions"])]
print(f"✓ Removed {before - len(policies)} logs-only policies")

# ── Step 3: Drop near-duplicate policies (same action set) ───────────────────
seen_action_sets = set()
unique_policies  = []

for p in policies:
    key = frozenset(p["actions"])
    if key not in seen_action_sets:
        seen_action_sets.add(key)
        unique_policies.append(p)

print(f"✓ Removed {len(policies) - len(unique_policies)} duplicate action sets")
print(f"Final count: {len(unique_policies)} policies")

# ── Step 4: Save cleaned version ─────────────────────────────────────────────
output_path = os.path.join(DRIVE_BASE, "data/processed/github_iam_policies_clean.json")
with open(output_path, "w") as f:
    json.dump(unique_policies, f, indent=2)

print(f"\n✓ Saved to: {output_path}")

# ── Step 5: Quick summary of what remains ────────────────────────────────────
from collections import Counter

print("\nRemaining policies:")
for i, p in enumerate(unique_policies):
    print(f"  [{i:02d}] {p['repo']:<55} "
          f"{len(p['actions']):>2} actions  "
          f"{p['resource_name']}")

print("\nAction count distribution:")
sizes = [len(p["actions"]) for p in unique_policies]
for bucket in [(2,4), (5,7), (8,10), (11,15), (16,25)]:
    count = sum(1 for s in sizes if bucket[0] <= s <= bucket[1])
    print(f"  {bucket[0]}-{bucket[1]} actions: {count} policies")

Starting count: 50 policies
✓ Deduplicated actions within each policy
✓ Removed 2 logs-only policies
✓ Removed 0 duplicate action sets
Final count: 48 policies

✓ Saved to: /content/drive/MyDrive/iam_rag_project/data/processed/github_iam_policies_clean.json

Remaining policies:
  [00] awslabs/ecs-refarch-continuous-deployment               14 actions  CodeBuildServiceRole
  [01] awslabs/ecs-refarch-continuous-deployment               13 actions  CodePipelineServiceRole
  [02] aws-samples/retail-demo-store                            7 actions  AlexaSkillIAMRole
  [03] aws-samples/retail-demo-store                            4 actions  AmazonPaySigningLambdaExecutionRole
  [04] aws-samples/aws-etl-orchestrator                        15 actions  AthenaRunnerPolicy
  [05] aws-samples/aws-etl-orchestrator                         4 actions  AWSGlueJobRole
  [06] aws-samples/serverless-samples                           8 actions  ApiGatewayInspectorProxyPolicy
  [07] aws-samples/serverless-sa

In [ ]:
import json
import os

input_path = os.path.join(DRIVE_BASE, "data/processed/github_iam_policies_clean.json")
with open(input_path) as f:
    policies = json.load(f)

# Fix 1 — correct dynamoDB prefix typo
for p in policies:
    p["actions"] = [
        a.replace("dynamoDB:", "dynamodb:") for a in p["actions"]
    ]
    p["policy_document"]["Statement"][0]["Action"] = p["actions"]

# Fix 2 — drop single-action policies
policies = [p for p in policies if len(p["actions"]) >= 2]

# Fix 3 — drop [36] InstanceRoleS3Policy (subset of [37])
# Keep the one with more actions when same resource_name appears twice
seen = {}
deduped = []
for p in policies:
    key = (p["repo"], p["resource_name"])
    if key not in seen:
        seen[key] = len(deduped)
        deduped.append(p)
    else:
        # Keep whichever has more actions
        existing_idx = seen[key]
        if len(p["actions"]) > len(deduped[existing_idx]["actions"]):
            deduped[existing_idx] = p

policies = deduped
print(f"Final count: {len(policies)} policies")

output_path = os.path.join(DRIVE_BASE, "data/processed/github_iam_policies_clean.json")
with open(output_path, "w") as f:
    json.dump(policies, f, indent=2)

print(f"✓ Saved to: {output_path}")

In [ ]:
# ── Step 8: Summary ───────────────────────────────────────────────────────────
from collections import Counter

print("\nService coverage:")
service_counts = Counter()
for p in unique_policies:
    for a in p["actions"]:
        if ":" in a:
            service_counts[a.split(":")[0]] += 1

for service, count in service_counts.most_common(15):
    print(f"  {service:<20} {count:>3} action references")

print("\nAction count distribution:")
sizes = [len(p["actions"]) for p in unique_policies]
for bucket in [(2,4), (5,7), (8,10), (11,15)]:
    count = sum(1 for s in sizes if bucket[0] <= s <= bucket[1])
    print(f"  {bucket[0]}-{bucket[1]} actions: {count} policies")

print("\nSample policy:")
if unique_policies:
    sample = unique_policies[0]
    print(f"  Repo:    {sample['repo']}")
    print(f"  Actions: {sample['actions']}")


Service coverage:
  logs                  58 action references
  s3                    52 action references
  dynamodb              22 action references
  ec2                   22 action references
  ecr                   18 action references
  sqs                   12 action references
  servicecatalog        12 action references
  states                 7 action references
  lambda                 7 action references
  kinesis                7 action references
  ecs                    6 action references
  cloudwatch             6 action references
  kms                    6 action references
  inspector              6 action references
  secretsmanager         5 action references

Action count distribution:
  2-4 actions: 18 policies
  5-7 actions: 17 policies
  8-10 actions: 2 policies
  11-15 actions: 10 policies

Sample policy:
  Repo:    awslabs/ecs-refarch-continuous-deployment
  Actions: ['ecr:BatchCheckLayerAvailability', 'ecr:BatchGetImage', 'ecr:CompleteLayerUpload', 'ecr

In [ ]:
#Form the second part of the ground truth by selecting high quality policies from manged_policies.json
import requests
import json
import time
import os
from bs4 import BeautifulSoup

TARGET_POLICIES = [
    # CodeDeploy specific patterns (avoid the wildcard ones)
    "AWSCodeDeployDeployerAccess",
    "AWSCodeDeployReadOnlyAccess",
    # Step Functions specific execution patterns
    "CloudWatchEventsBuiltInTargetExecutionAccess",
    "AWSGlueSchemaRegistryReadonlyAccess",
    # SNS specific
    "AmazonSNSRole",
    "AmazonSNSFullAccess",
]

BASE    = "https://docs.aws.amazon.com/aws-managed-policy/latest/reference/"
headers = {"User-Agent": "Mozilla/5.0"}

fetched_policies = []

print("=" * 60)
print("FETCHING TARGET MANAGED POLICIES")
print("=" * 60)

for policy_name in TARGET_POLICIES:
    url = BASE + policy_name + ".html"
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            print(f"✗ {policy_name:<45} HTTP {r.status_code}")
            continue

        page        = BeautifulSoup(r.content, "html.parser")
        policy_json = None

        # Primary: <pre> tag
        pre = page.find("pre")
        if pre:
            try:
                policy_json = json.loads(pre.text.strip())
            except json.JSONDecodeError:
                pass

        # Fallback: <code> tag
        if not policy_json:
            for code in page.find_all("code"):
                text = code.text.strip()
                if text.startswith("{") and "Version" in text:
                    try:
                        policy_json = json.loads(text)
                        break
                    except json.JSONDecodeError:
                        continue

        if not policy_json or "Statement" not in policy_json:
            print(f"✗ {policy_name:<45} could not parse JSON")
            continue

        # Extract all actions
        all_actions = []
        for stmt in policy_json.get("Statement", []):
            raw = stmt.get("Action", [])
            if isinstance(raw, str):
                raw = [raw]
            all_actions.extend(raw)

        all_actions = list(set(all_actions))

        # Classify actions
        specific  = [a for a in all_actions if isinstance(a, str)
                     and ':' in a and '*' not in a]
        wildcards = [a for a in all_actions if isinstance(a, str)
                     and '*' in a]
        services  = sorted({
            a.split(':')[0].lower()
            for a in all_actions
            if isinstance(a, str) and ':' in a
        })

        fetched_policies.append({
            "policy_name":     policy_name,
            "url":             url,
            "policy_document": policy_json,
            "actions_used":    all_actions,
        })

        # Print quality summary
        wc_str = f" | wildcards: {wildcards}" if wildcards else ""
        print(f"✓ {policy_name}")
        print(f"  services:  {services}")
        print(f"  specific:  {len(specific)} actions")
        print(f"  all:       {sorted(specific)[:8]}"
              f"{'...' if len(specific) > 8 else ''}")
        if wildcards:
            print(f"  wildcards: {wildcards}")
        print()

        time.sleep(0.5)

    except Exception as e:
        print(f"✗ {policy_name:<45} ERROR: {e}")

print(f"Successfully fetched: {len(fetched_policies)} / {len(TARGET_POLICIES)} policies")

FETCHING TARGET MANAGED POLICIES
✓ AWSCodeDeployDeployerAccess
  services:  ['chatbot', 'codedeploy', 'codestar-notifications', 'sns']
  specific:  13 actions
  all:       ['chatbot:DescribeSlackChannelConfigurations', 'codedeploy:CreateDeployment', 'codedeploy:RegisterApplicationRevision', 'codestar-notifications:CreateNotificationRule', 'codestar-notifications:DescribeNotificationRule', 'codestar-notifications:ListEventTypes', 'codestar-notifications:ListNotificationRules', 'codestar-notifications:ListTagsforResource']...
  wildcards: ['codedeploy:Batch*', 'codedeploy:Get*', 'codedeploy:List*']

✓ AWSCodeDeployReadOnlyAccess
  services:  ['codedeploy', 'codestar-notifications']
  specific:  4 actions
  all:       ['codestar-notifications:DescribeNotificationRule', 'codestar-notifications:ListEventTypes', 'codestar-notifications:ListNotificationRules', 'codestar-notifications:ListTargets']
  wildcards: ['codedeploy:Batch*', 'codedeploy:Get*', 'codedeploy:List*']

✓ CloudWatchEventsBui

In [ ]:
import json
import os
from collections import Counter

with open(os.path.join(DRIVE_BASE, 'data/processed/managed_policies.json')) as f:
    managed = json.load(f)

# Services in your GitHub set for reference
github_covered = {
    'logs', 's3', 'dynamodb', 'ecr', 'ec2', 'sqs', 'lambda',
    'ecs', 'cloudwatch', 'kms', 'secretsmanager', 'cloudformation',
    'ssm', 'codebuild', 'iam', 'xray', 'apigateway', 'states',
    'kinesis', 'redshift-data', 'athena', 'redshift', 'apigatewayv2',
    'events', 'sns', 'firehose', 'codedeploy', 'elasticloadbalancing',
    'rekognition', 'ses', 'bedrock'
}

def get_specific_actions(policy):
    return [
        a for a in policy.get('actions_used', [])
        if isinstance(a, str) and ':' in a and '*' not in a
    ]

def get_wildcards(policy):
    return [
        a for a in policy.get('actions_used', [])
        if isinstance(a, str) and '*' in a
    ]

def get_services(policy):
    services = set()
    for a in policy.get('actions_used', []):
        if isinstance(a, str) and ':' in a:
            services.add(a.split(':')[0].lower())
    return services

def score_policy(policy):
    specific  = get_specific_actions(policy)
    wildcards = get_wildcards(policy)
    services  = get_services(policy)

    # ── Hard filters ─────────────────────────────────────────────────
    if len(specific) < 3:
        return 0
    if len(specific) > 20:
        return 0
    service_wildcards = [w for w in wildcards if w.endswith(':*')]
    if service_wildcards:
        return 0

    # ── Base score ───────────────────────────────────────────────────
    score = 10

    # ── Core target service bonus ─────────────────────────────────────
    core_targets = {
        's3', 'lambda', 'dynamodb', 'ec2', 'ecs', 'ecr',
        'sqs', 'sns', 'logs', 'cloudwatch', 'events',
        'kms', 'secretsmanager', 'ssm', 'iam', 'sts',
        'codedeploy', 'codepipeline', 'glue', 'states',
        'kinesis', 'firehose', 'rds', 'eks', 'kafka',
        'redshift', 'athena', 'codebuild', 'xray'
    }
    score += len(services & core_targets) * 2

    # ── Priority missing service bonus ────────────────────────────────
    priority = {
        'codedeploy', 'codepipeline', 'glue', 'states',
        'firehose', 'events', 'sns', 'kinesis', 'kafka',
        'rds', 'eks', 'redshift', 'athena'
    }
    score += len(services & priority) * 5

    # ── Relevant novel service bonus ──────────────────────────────────
    # Only reward novel services that are actually useful
    novel_services = services - github_covered
    relevant_novel = {
        'rds', 'eks', 'autoscaling', 'kafka', 'msk',
        'ecr-public', 'glue', 'codepipeline', 'codedeploy',
        'states', 'firehose', 'kinesis', 'events', 'sns',
        'stepfunctions', 's3-object-lambda', 'oam', 'redshift'
    }
    score += len(novel_services & relevant_novel) * 5
    score += len(novel_services - relevant_novel) * 1

    # ── Action count sweet spot bonus ────────────────────────────────
    n = len(specific)
    if 4 <= n <= 12:
        score += 4
    elif 13 <= n <= 20:
        score += 2

    # ── Wildcard penalty ─────────────────────────────────────────────
    score -= len(wildcards) * 2

    # ── Too many services penalty ─────────────────────────────────────
    if len(services) > 6:
        score -= (len(services) - 6) * 2

    # ── Niche service penalty ─────────────────────────────────────────
    niche = {
        'chatbot', 'codestar-notifications', 'databrew',
        'opsworks', 'mobiletargeting', 'geo', 'monitron',
        'codeguru', 'devopsguru', 'inspector', 'datasync',
        'dms', 'mgn', 'thinclient', 'workspaces',
        'workspaces-web', 'aiops', 'identitystore',
        'cassandra', 'kinesisanalytics', 'rum', 'evidently',
        'imagebuilder', 'resource-explorer-2',
        'application-signals', 'fsx', 'sdb', 'outposts',
        's3-outposts', 'drs', 'codeconnections',
        'codestar-connections', 'guardduty', 'pi',
        'elasticmapreduce', 'servicecatalog', 'appstream',
        'aoss', 'es', 'opensearch', 'synthetics',
        'cloudfront', 'organizations', 'account',
        'sso', 'sso-directory', 'elasticbeanstalk'
    }
    niche_overlap = services & niche
    score -= len(niche_overlap) * 5

    return max(score, 0)


# Score all managed policies
scored = []
for p in managed:
    s = score_policy(p)
    if s > 0:
        scored.append((p, s))

scored.sort(key=lambda x: x[1], reverse=True)

print(f"Total managed policies available:  {len(managed)}")
print(f"Policies passing hard filters:     {len(scored)}")
print(f"\nTop 50 candidates:")
print(f"{'score':>7}  {'policy_name':<58} {'services':<8} {'specific':>9} {'wildcards':>10}")
print("-" * 100)

for p, score in scored[:50]:
    services   = get_services(p)
    specific   = get_specific_actions(p)
    wildcards  = get_wildcards(p)
    priority   = {'codedeploy','codepipeline','glue','states',
                  'firehose','events','sns','kinesis'}
    priority_hit = services & priority
    novel      = services - github_covered

    extras = []
    if priority_hit:
        extras.append(f"priority={priority_hit}")
    if novel:
        extras.append(f"novel={novel}")
    extras_str = " | ".join(extras)

    print(f"  score={score:>3}  {p['policy_name']:<58} "
          f"svc={len(services):<4} "
          f"specific={len(specific):<5} "
          f"wc={len(wildcards):<4} "
          f"{extras_str}")

Total managed policies available:  500
Policies passing hard filters:     205

Top 50 candidates:
  score  policy_name                                                services  specific  wildcards
----------------------------------------------------------------------------------------------------
  score= 30  AWSLambdaMSKExecutionRole                                  svc=3    specific=12    wc=0    novel={'kafka'}
  score= 30  AWSApplicationAutoscalingRDSClusterPolicy                  svc=3    specific=10    wc=0    novel={'rds'}
  score= 30  AWSBudgetsActions_RolePolicyForResourceAdministrationWithSSM svc=3    specific=7     wc=0    novel={'rds'}
  score= 29  AmazonRDSReadOnlyAccess                                    svc=5    specific=16    wc=1    novel={'rds', 'devops-guru'}
  score= 27  AWSCodeDeployRoleForLambda                                 svc=4    specific=8     wc=0    priority={'sns'}
  score= 27  AWSFaultInjectionSimulatorRDSAccess                        svc=2    specific=5

In [ ]:
import json
import os

# ── The 27 selected from scoring + 6 from targeted fetch ─────────────────────

SELECTED_FROM_SCORING = [
    "AWSLambdaMSKExecutionRole",
    "AWSApplicationAutoscalingRDSClusterPolicy",
    "AWSCodeDeployRoleForLambda",
    "AmazonSSMMaintenanceWindowRole",
    "AWSCodeDeployRoleForECS",
    "AmazonOpenSearchDirectQueryGlueCreateAccess",
    "AWSLambdaKinesisExecutionRole",
    "DynamoDBKinesisReplicationServiceRolePolicy",
    "AWSElasticBeanstalkRoleSNS",
    "AmazonEKSDashboardConsoleReadOnly",
    "AmazonElasticContainerRegistryPublicReadOnly",
    "AmazonS3ObjectLambdaExecutionRolePolicy",
    "AWSCodeDeployRoleForLambdaLimited",
    "AWSLambdaDynamoDBExecutionRole",
    "AWSLambdaSQSQueueExecutionRole",
    "AWSLambdaInvocation-DynamoDB",
    "AWSTransformSecretsManagerConnectorPolicy",
    "AWSFaultInjectionSimulatorEC2Access",
    "AWSFaultInjectionSimulatorECSAccess",
    "CloudWatchLogsCrossAccountSharingConfiguration",
    "AmazonEC2SpotFleetAutoscaleRole",
    "AWSElasticBeanstalkRoleRDS",
    "CloudWatchLambdaApplicationSignalsExecutionRolePolicy",
    "AmazonRedshiftDataFullAccess",
    "AWSEC2SqlHaServiceRolePolicy",
    "AWS-SSM-DiagnosisAutomation-AdministrationRolePolicy",
    "CloudWatchAgentServerPolicy",
]

SELECTED_FROM_TARGETED = [
    "AWSCodeDeployRoleForLambda",      # already in scoring list — dedup handled below
    "AWSCodeDeployRoleForECS",         # already in scoring list — dedup handled below
    "AWSCodePipeline_ReadOnlyAccess",
    "AWSStepFunctionsReadOnlyAccess",
    "AWSGlueSchemaRegistryReadonlyAccess",
]

# Combine and deduplicate by policy name
all_selected_names = list(dict.fromkeys(
    SELECTED_FROM_SCORING + SELECTED_FROM_TARGETED
))
print(f"Total unique selected policy names: {len(all_selected_names)}")


# ── Load managed policies and extract selected ones ───────────────────────────
managed_path = os.path.join(DRIVE_BASE, "data/processed/managed_policies.json")
with open(managed_path) as f:
    managed = json.load(f)

# Build lookup by policy name
managed_lookup = {p["policy_name"]: p for p in managed}

found     = []
not_found = []

for name in all_selected_names:
    if name in managed_lookup:
        found.append(managed_lookup[name])
    else:
        not_found.append(name)

print(f"Found in managed_policies.json:  {len(found)}")
print(f"Not found (need to fetch):       {len(not_found)}")
if not_found:
    print(f"  Missing: {not_found}")


# ── Fetch any missing policies directly from AWS docs ─────────────────────────
if not_found:
    import requests
    import time
    from bs4 import BeautifulSoup

    BASE    = "https://docs.aws.amazon.com/aws-managed-policy/latest/reference/"
    headers = {"User-Agent": "Mozilla/5.0"}

    print(f"\nFetching {len(not_found)} missing policies...")
    for policy_name in not_found:
        url = BASE + policy_name + ".html"
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code != 200:
                print(f"  ✗ {policy_name} — HTTP {r.status_code}")
                continue

            page        = BeautifulSoup(r.content, "html.parser")
            policy_json = None

            pre = page.find("pre")
            if pre:
                try:
                    policy_json = json.loads(pre.text.strip())
                except json.JSONDecodeError:
                    pass

            if not policy_json:
                for code in page.find_all("code"):
                    text = code.text.strip()
                    if text.startswith("{") and "Version" in text:
                        try:
                            policy_json = json.loads(text)
                            break
                        except json.JSONDecodeError:
                            continue

            if not policy_json or "Statement" not in policy_json:
                print(f"  ✗ {policy_name} — could not parse")
                continue

            actions_used = []
            for stmt in policy_json.get("Statement", []):
                raw = stmt.get("Action", [])
                if isinstance(raw, str):
                    raw = [raw]
                actions_used.extend(raw)

            fetched = {
                "policy_name":     policy_name,
                "url":             url,
                "policy_document": policy_json,
                "actions_used":    list(set(actions_used)),
            }
            found.append(fetched)
            print(f"  ✓ {policy_name} — {len(actions_used)} actions")
            time.sleep(0.5)

        except Exception as e:
            print(f"  ✗ {policy_name} — ERROR: {e}")


# ── Build ground truth policy set ────────────────────────────────────────────
def get_specific_actions(policy):
    return sorted(list(set([
        a for a in policy.get("actions_used", [])
        if isinstance(a, str) and ":" in a and "*" not in a
    ])))

ground_truth_managed = []

for p in found:
    specific = get_specific_actions(p)
    services = sorted({
        a.split(":")[0].lower()
        for a in specific if ":" in a
    })

    ground_truth_managed.append({
        "policy_name":     p["policy_name"],
        "source":          "aws_managed",
        "url":             p.get("url", ""),
        "policy_document": p["policy_document"],
        "actions":         specific,
        "services":        services,
        "action_count":    len(specific),
    })

print(f"\n✓ Built {len(ground_truth_managed)} managed policy ground truth entries")


# ── Load GitHub ground truth ──────────────────────────────────────────────────
github_path = os.path.join(
    DRIVE_BASE, "data/evaluation/github_iam_policies_final.json"
)
with open(github_path) as f:
    github_policies = json.load(f)

# Standardize GitHub format to match managed format
ground_truth_github = []
for p in github_policies:
    ground_truth_github.append({
        "policy_name":     p.get("resource_name", "unknown"),
        "source":          "github",
        "url":             p.get("source_url", ""),
        "repo":            p.get("repo", ""),
        "policy_document": p["policy_document"],
        "actions":         sorted(list(set(p["actions"]))),
        "services":        sorted({
            a.split(":")[0].lower()
            for a in p["actions"] if ":" in a
        }),
        "action_count":    len(p["actions"]),
    })

print(f"✓ Loaded {len(ground_truth_github)} GitHub ground truth entries")


# ── Combine into final evaluation set ────────────────────────────────────────
ground_truth_eval = ground_truth_github + ground_truth_managed

print(f"\n{'='*50}")
print(f"FINAL EVALUATION SET: {len(ground_truth_eval)} policies")
print(f"  GitHub:  {len(ground_truth_github)}")
print(f"  Managed: {len(ground_truth_managed)}")
print(f"{'='*50}")


# ── Summary statistics ────────────────────────────────────────────────────────
from collections import Counter

print("\nAction count distribution:")
sizes = [p["action_count"] for p in ground_truth_eval]
for bucket in [(2,4), (5,8), (9,12), (13,20)]:
    count = sum(1 for s in sizes if bucket[0] <= s <= bucket[1])
    print(f"  {bucket[0]}-{bucket[1]} actions: {count} policies")

print("\nService coverage across full eval set:")
service_counts = Counter()
for p in ground_truth_eval:
    for svc in p["services"]:
        service_counts[svc] += 1
for svc, count in service_counts.most_common(20):
    print(f"  {svc:<25} {count:>3} policies")

print("\nSource breakdown:")
sources = Counter(p["source"] for p in ground_truth_eval)
for src, count in sources.items():
    print(f"  {src:<15} {count} policies")


# ── Save ──────────────────────────────────────────────────────────────────────
output_path = os.path.join(
    DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json"
)
with open(output_path, "w") as f:
    json.dump(ground_truth_eval, f, indent=2)

print(f"\n✓ Saved evaluation set to: {output_path}")

Total unique selected policy names: 30
Found in managed_policies.json:  27
Not found (need to fetch):       3
  Missing: ['AWSCodePipeline_ReadOnlyAccess', 'AWSStepFunctionsReadOnlyAccess', 'AWSGlueSchemaRegistryReadonlyAccess']

Fetching 3 missing policies...
  ✓ AWSCodePipeline_ReadOnlyAccess — 16 actions
  ✓ AWSStepFunctionsReadOnlyAccess — 15 actions
  ✓ AWSGlueSchemaRegistryReadonlyAccess — 11 actions

✓ Built 30 managed policy ground truth entries
✓ Loaded 45 GitHub ground truth entries

FINAL EVALUATION SET: 75 policies
  GitHub:  45
  Managed: 30

Action count distribution:
  2-4 actions: 23 policies
  5-8 actions: 26 policies
  9-12 actions: 14 policies
  13-20 actions: 11 policies

Service coverage across full eval set:
  logs                       23 policies
  s3                         22 policies
  dynamodb                   11 policies
  lambda                     11 policies
  ec2                        10 policies
  ssm                         9 policies
  iam         

In [ ]:
# ── Generate natural language queries for all 76 ground truth policies ─────────
# Uses GPT-4o to generate queries without mentioning service/action names
# Following Gorilla methodology — query describes the task, not the API

import json
import os
import time

import openai
from google.colab import userdata

client = openai.OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)


# Load ground truth
gt_path = os.path.join(DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json")
with open(gt_path) as f:
    ground_truth = json.load(f)

print(f"Generating queries for {len(ground_truth)} policies...")
print()

# ── Updated system prompt ─────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a developer writing a task description that will be used
to generate an AWS IAM policy. Given an IAM policy, write a clear, specific task
description that captures exactly what permissions are needed and why.

RULES:
- Do NOT mention specific AWS service names (e.g. do not say "S3", "Lambda",
  "DynamoDB", "SQS", "ECR", "CloudWatch", "EventBridge", "SSM", "KMS", etc.)
- Do NOT mention specific IAM action names (e.g. do not say "GetObject",
  "PutItem", "InvokeFunction", "CreateLogGroup", etc.)
- Use a descriptive subject that reflects the role's purpose
   (e.g. 'Allow a container build process to...' not just 'Allow a function to...')
   but do not use AWS service names as the subject.
- Focus only on primary functional tasks. Do not describe supporting
   infrastructure permissions such as network interface management,
   resource tagging, or identity assumption as separate tasks unless
   they are the main purpose of the policy.
- If the policy includes role-passing permissions, do not mention them
   as a separate task — they are implicit supporting permissions.
   Only describe the primary functional tasks.
- Write as a statement, not a question
- Do not wrap the query in quotation marks
- Be specific about WHAT the service/role needs to do, not just that it needs access
- Mention the different functional areas covered if the policy spans multiple tasks
- Keep to 1-3 sentences

GOOD EXAMPLES:

Policy actions: s3:GetObject, s3:ListBucket, s3:GetBucketLocation
Good query: Allow read-only access to retrieve objects and list contents of a storage bucket

Policy actions: sqs:ReceiveMessage, sqs:DeleteMessage, sqs:GetQueueAttributes, dynamodb:PutItem, logs:CreateLogGroup, logs:CreateLogStream, logs:PutLogEvents
Good query: Allow a function to consume messages from a queue, store processed results in a database table, and write execution logs

Policy actions: ecr:GetAuthorizationToken, ecr:BatchGetImage, ecr:GetDownloadUrlForLayer, ecr:BatchCheckLayerAvailability, logs:CreateLogStream, logs:PutLogEvents
Good query: Allow a container runtime to authenticate and pull container images from a private registry and write task execution logs

Policy actions: codedeploy:CreateDeployment, codedeploy:GetDeployment, codedeploy:GetDeploymentConfig, ecs:DescribeServices, ecs:UpdateService
Good query: Allow a deployment service to create and monitor deployments and update the target container service during rollout

BAD EXAMPLES:

Too vague:
  Allow the ability to trigger specific functions related to account inspection
  → Does not describe the actual task clearly enough

Mentions service names:
  Allow Lambda to read from SQS and write to DynamoDB
  → SQS, Lambda, DynamoDB are service names

Question format:
  How can I enable read access to my storage bucket?
  → Should be a statement not a question

Too generic:
  Allow full access to manage files in a storage bucket
  → "Full access" is vague — specify the operations (read, write, delete, list)

IMPORTANT: The query must be specific enough that someone reading it would know
exactly which functional areas need permissions. If the policy covers multiple
services doing different things, mention each distinct task."""


def generate_query(policy_entry):
    """
    Generate a natural language query for a single ground truth policy.
    Returns the query string or None on failure.
    """
    policy_doc = policy_entry["policy_document"]
    actions    = policy_entry["actions"]
    source     = policy_entry["source"]

    # Build context
    context_parts = []

    if source == "github":
        resource_name = policy_entry.get("policy_name", "")
        repo          = policy_entry.get("repo", "")
        context_parts.append(f"CloudFormation resource name: {resource_name}")
        context_parts.append(f"Source repository: {repo}")
    else:
        policy_name = policy_entry.get("policy_name", "")
        context_parts.append(f"AWS managed policy name: {policy_name}")

    context_parts.append(
        f"\nIAM Policy Document:\n{json.dumps(policy_doc, indent=2)}"
    )
    context_parts.append(
        f"\nSpecific actions granted: {', '.join(actions)}"
    )

    # Add decomposition hint for multi-service policies
    services = policy_entry.get("services", [])
    if len(services) > 2:
        context_parts.append(
            f"\nThis policy covers {len(services)} different service areas: "
            f"{', '.join(services)}. Make sure the query describes all "
            f"distinct functional tasks covered."
        )

    context_parts.append(
        "\n\nWrite a task description for this policy following the rules above."
    )

    user_message = "\n".join(context_parts)

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_message}
            ],
            temperature=0.2,  # lower than before — more consistent outputs
            max_tokens=200    # slightly more room for multi-service policies
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"  Error: {e}")
        return None

# ── Run query generation ──────────────────────────────────────────────────────
failed  = []
success = 0

for i, entry in enumerate(ground_truth):
    policy_name = entry.get("policy_name", f"policy_{i}")
    source      = entry.get("source", "unknown")

    print(f"[{i+1:02d}/{len(ground_truth)}] {policy_name} ({source})")

    query = generate_query(entry)

    if query:
        entry["query"] = query
        success += 1
        print(f"  → {query}")
    else:
        failed.append(i)
        print(f"  → FAILED")

    # Respect rate limits — GPT-4o allows ~500 RPM on tier 1
    time.sleep(0.5)

    # Save progress every 10 policies in case of interruption
    if (i + 1) % 10 == 0:
        with open(gt_path, "w") as f:
            json.dump(ground_truth, f, indent=2)
        print(f"  [checkpoint saved at {i+1} policies]")


# ── Final save ────────────────────────────────────────────────────────────────
with open(gt_path, "w") as f:
    json.dump(ground_truth, f, indent=2)

print()
print(f"{'='*55}")
print(f"Query generation complete")
print(f"  Success: {success} / {len(ground_truth)}")
print(f"  Failed:  {len(failed)}")
if failed:
    print(f"  Failed indices: {failed}")
print(f"{'='*55}")

# ── Print sample queries for review ──────────────────────────────────────────
print("\nSample queries (first 10):")
for i, entry in enumerate(ground_truth[:10]):
    print(f"\n  [{i:02d}] {entry.get('policy_name')}")
    print(f"       Actions:  {entry['actions'][:4]}{'...' if len(entry['actions'])>4 else ''}")
    print(f"       Query:    {entry.get('query', 'NOT GENERATED')}")

Generating queries for 75 policies...

[01/75] CodeBuildServiceRole (github)
  → Allow a build process to authenticate and manage container images in a private registry, including checking layer availability, uploading, and retrieving images. Enable the build process to write execution logs for monitoring purposes. Permit access to retrieve and store objects in a storage service for build artifacts and dependencies.
[02/75] CodePipelineServiceRole (github)
  → Allow a deployment pipeline to initiate and monitor build processes, manage and update container task definitions and services, retrieve and store objects in a storage bucket, and pass roles for execution.
[03/75] AmazonPaySigningLambdaExecutionRole (github)
  → Allow a function to retrieve sensitive configuration data from a secure storage service and write execution logs for monitoring and troubleshooting purposes.
[04/75] AthenaRunnerPolicy (github)
  → Allow an ETL orchestration process to execute and monitor data queries, ma

In [13]:
#remove the 30 ground truth policies from the policy retrieval corpus
import json
import os
import numpy as np
import faiss

# ── Load evaluation policy names to remove ───────────────────────────────────
gt_path = os.path.join(DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json")
with open(gt_path) as f:
    ground_truth = json.load(f)

gt_managed_names = {
    p["policy_name"]
    for p in ground_truth
    if p["source"] == "aws_managed"
}
print(f"Policy embeddings to remove: {len(gt_managed_names)}")

# ── Load original policy embeddings ──────────────────────────────────────────
policy_emb_path = os.path.join(
    DRIVE_BASE, "indexes/dense/policy_chunks_embedded.json"
)
policy_emb_cleaned_path = os.path.join(
    DRIVE_BASE, "indexes/dense/policy_chunks_embedded_cleaned.json"
)
with open(policy_emb_path) as f:
    policy_chunks = json.load(f)

print(f"Policy chunks before removal: {len(policy_chunks)}")

# ── Filter out evaluation policies ───────────────────────────────────────────
clean_policy_chunks = [
    chunk for chunk in policy_chunks
    if chunk["metadata"].get("policy_name") not in gt_managed_names
]

removed = len(policy_chunks) - len(clean_policy_chunks)
print(f"Removed:                  {removed} chunks")
print(f"Policy chunks remaining:  {len(clean_policy_chunks)}")

# Verify
removed_names = {
    chunk["metadata"].get("policy_name")
    for chunk in policy_chunks
    if chunk["metadata"].get("policy_name") in gt_managed_names
}
not_found = gt_managed_names - removed_names
if not_found:
    print(f"NOTE — not found in embeddings (expected if manually fetched): {not_found}")
else:
    print(f"✓ All {removed} evaluation policies removed from embeddings")

# ── Save cleaned policy embeddings ───────────────────────────────────────────
with open(policy_emb_cleaned_path, "w") as f:
    json.dump(clean_policy_chunks, f)
print(f"✓ Saved policy_chunks_embedded_cleaned.json")

# ── Rebuild combined index in original format ─────────────────────────────────
action_emb_path = os.path.join(
    DRIVE_BASE, "indexes/dense/action_chunks_embedded.json"
)
with open(action_emb_path) as f:
    action_chunks = json.load(f)

# Combine — order matches original: actions first, then policies
combined = action_chunks + clean_policy_chunks

print(f"\nCombining dense chunks:")
print(f"  Action chunks:  {len(action_chunks)}")
print(f"  Policy chunks:  {len(clean_policy_chunks)}")
print(f"  Combined total: {len(combined)}")

# ── Save combined_chunks.json — same format as original ──────────────────────
combined_chunks_path = os.path.join(
    DRIVE_BASE, "indexes/dense/combined_chunks.json"
)
with open(combined_chunks_path, "w") as f:
    json.dump(combined, f)
print(f"  ✓ Saved combined_chunks.json")

# ── Build FAISS index ─────────────────────────────────────────────────────────
vectors = np.array(
    [c["embedding"] for c in combined],
    dtype="float32"
)
print(f"  Vector matrix shape: {vectors.shape}")

# Normalize for cosine similarity — same as original
faiss.normalize_L2(vectors)

dim   = vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(vectors)

# ── Save as .index — same filename and extension as original ──────────────────
faiss_index_path = os.path.join(
    DRIVE_BASE, "indexes/dense/combined_faiss.index"
)
faiss.write_index(index, faiss_index_path)

print(f"  ✓ FAISS index saved: {faiss_index_path}")
print(f"  ✓ Total vectors in index: {index.ntotal}")

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"\nSanity check — verifying removed policies not in combined_chunks.json:")
leakage_found = False
for chunk in combined:
    name = chunk["metadata"].get("policy_name", "")
    if name in gt_managed_names:
        print(f"  ✗ LEAKAGE: {name} still present")
        leakage_found = True

if not leakage_found:
    print(f"  ✓ No leakage — all evaluation policies absent from index")

print(f"\nIndex rebuild complete:")
print(f"  Action chunks:  {len(action_chunks)}")
print(f"  Policy chunks:  {len(clean_policy_chunks)}")
print(f"  Total:          {index.ntotal}")

Policy embeddings to remove: 30
Policy chunks before removal: 500
Removed:                  27 chunks
Policy chunks remaining:  473
NOTE — not found in embeddings (expected if manually fetched): {'AWSCodePipeline_ReadOnlyAccess', 'AWSGlueSchemaRegistryReadonlyAccess', 'AWSStepFunctionsReadOnlyAccess'}
✓ Saved policy_chunks_embedded_cleaned.json

Combining dense chunks:
  Action chunks:  4505
  Policy chunks:  473
  Combined total: 4978
  ✓ Saved combined_chunks.json
  Vector matrix shape: (4978, 1024)
  ✓ FAISS index saved: /content/drive/MyDrive/iam_rag_project/indexes/dense/combined_faiss.index
  ✓ Total vectors in index: 4978

Sanity check — verifying removed policies not in combined_chunks.json:
  ✓ No leakage — all evaluation policies absent from index

Index rebuild complete:
  Action chunks:  4505
  Policy chunks:  473
  Total:          4978


In [6]:
import json
import os
from openai import OpenAI

from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

# ── Load ground truth ─────────────────────────────────────────────────────────
gt_path = os.path.join(DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json")
with open(gt_path) as f:
    ground_truth = json.load(f)

# ── Split by source and sample 10 from each ──────────────────────────────────
github_policies  = [p for p in ground_truth if p["source"] == "github"]
managed_policies = [p for p in ground_truth if p["source"] == "aws_managed"]

# Take first 10 from each — deterministic, no random sampling
github_sample  = github_policies[:10]
managed_sample = managed_policies[:10]

print(f"GitHub policies available:  {len(github_policies)}")
print(f"Managed policies available: {len(managed_policies)}")
print(f"Sampling 10 from each\n")

# ── Generator prompt ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an AWS IAM policy expert. Given a task description,
generate a valid AWS IAM policy JSON that grants exactly the permissions needed
to accomplish the task.

Rules:
- Output ONLY valid JSON, no explanation or markdown
- Follow the standard IAM policy structure with Version and Statement
- Use specific actions, not broad wildcards like s3:* unless truly necessary
- Set Resource to "*" unless a more specific resource is implied
- Include only the minimum permissions needed for the task
- The policy must be parseable as JSON"""

def generate_policy(query):
    """Generate an IAM policy for a natural language query using GPT-3.5."""
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Generate an IAM policy for this task:\n\n{query}"}
            ],
            temperature=0,
            max_tokens=1000
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {e}"


def extract_actions(policy_str):
    """Extract all Allow actions from a generated policy string."""
    try:
        # Strip markdown code blocks if present
        clean = policy_str.strip()
        if clean.startswith("```"):
            lines = clean.split("\n")
            clean = "\n".join(lines[1:-1])

        policy = json.loads(clean)
        actions = []
        for stmt in policy.get("Statement", []):
            if stmt.get("Effect") != "Allow":
                continue
            raw = stmt.get("Action", [])
            if isinstance(raw, str):
                raw = [raw]
            actions.extend(raw)
        return sorted(set(actions)), policy, None

    except json.JSONDecodeError as e:
        return [], None, f"JSON parse error: {e}"


def evaluate_basic(generated_actions, gt_actions):
    """Basic precision/recall/miss calculation."""
    gen_set = set(generated_actions)
    gt_set  = set(gt_actions)

    # Handle wildcards simply — if generated has s3:* count all gt s3 actions as covered
    covered = set()
    for ga in gen_set:
        if '*' in ga:
            prefix = ga.split(':')[0].lower()
            suffix = ga.split(':')[1]
            for gta in gt_set:
                gta_prefix = gta.split(':')[0].lower()
                if gta_prefix == prefix:
                    if suffix == '*' or gta.split(':')[1].startswith(suffix.replace('*','')):
                        covered.add(gta)
        else:
            if ga in gt_set:
                covered.add(ga)

    missing  = gt_set - covered
    extra    = gen_set - gt_set
    hallucinated = [
        a for a in gen_set
        if '*' not in a and a not in gt_set
    ]

    precision = len(covered) / len(gen_set)  if gen_set  else 0
    recall    = len(covered) / len(gt_set)   if gt_set   else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)

    return {
        "precision":    round(precision, 3),
        "recall":       round(recall, 3),
        "f1":           round(f1, 3),
        "missing":      sorted(missing),
        "extra":        sorted(extra),
        "n_generated":  len(gen_set),
        "n_expected":   len(gt_set),
        "n_covered":    len(covered),
    }


# ── Run generation and evaluation ─────────────────────────────────────────────
def run_sample(sample, label):
    print(f"\n{'='*65}")
    print(f"{label.upper()} — {len(sample)} POLICIES")
    print(f"{'='*65}")

    results = []
    total_precision = total_recall = total_f1 = 0
    parse_failures  = 0

    for i, entry in enumerate(sample):
        policy_name = entry["policy_name"]
        query       = entry["query"]
        gt_actions  = entry["actions"]

        print(f"\n[{i+1:02d}] {policy_name}")
        print(f"  Query:    {query[:90]}{'...' if len(query)>90 else ''}")
        print(f"  Expected: {gt_actions[:4]}{'...' if len(gt_actions)>4 else ''} ({len(gt_actions)} total)")

        # Generate
        generated_str = generate_policy(query)

        # Extract actions
        generated_actions, parsed_policy, error = extract_actions(generated_str)

        if error:
            print(f"  ✗ Parse failed: {error}")
            print(f"  Raw output: {generated_str[:200]}")
            parse_failures += 1
            results.append({
                "policy_name": policy_name,
                "query":       query,
                "gt_actions":  gt_actions,
                "error":       error,
                "generated_raw": generated_str
            })
            continue

        # Evaluate
        metrics = evaluate_basic(generated_actions, gt_actions)

        print(f"  Generated: {generated_actions[:4]}{'...' if len(generated_actions)>4 else ''} ({len(generated_actions)} total)")
        print(f"  Precision: {metrics['precision']:.3f} | "
              f"Recall: {metrics['recall']:.3f} | "
              f"F1: {metrics['f1']:.3f}")

        if metrics["missing"]:
            print(f"  Missing:  {metrics['missing']}")
        if metrics["extra"]:
            print(f"  Extra:    {metrics['extra'][:4]}{'...' if len(metrics['extra'])>4 else ''}")

        total_precision += metrics["precision"]
        total_recall    += metrics["recall"]
        total_f1        += metrics["f1"]

        results.append({
            "policy_name":      policy_name,
            "query":            query,
            "gt_actions":       gt_actions,
            "generated_actions": generated_actions,
            "generated_policy": parsed_policy,
            "metrics":          metrics
        })

    # Summary
    n_success = len(sample) - parse_failures
    print(f"\n{'─'*65}")
    print(f"{label} SUMMARY ({n_success}/{len(sample)} parsed successfully):")
    if n_success > 0:
        print(f"  Avg Precision: {total_precision/n_success:.3f}")
        print(f"  Avg Recall:    {total_recall/n_success:.3f}")
        print(f"  Avg F1:        {total_f1/n_success:.3f}")
    if parse_failures:
        print(f"  Parse failures: {parse_failures}")

    return results


# ── Run both samples ──────────────────────────────────────────────────────────
github_results  = run_sample(github_sample,  "GitHub Policies")
managed_results = run_sample(managed_sample, "Managed Policies")

# ── Overall summary ───────────────────────────────────────────────────────────
all_results = github_results + managed_results
all_metrics = [r["metrics"] for r in all_results if "metrics" in r]

print(f"\n{'='*65}")
print(f"OVERALL VANILLA GPT-3.5 PERFORMANCE (n={len(all_metrics)})")
print(f"{'='*65}")
if all_metrics:
    print(f"  Avg Precision: {sum(m['precision'] for m in all_metrics)/len(all_metrics):.3f}")
    print(f"  Avg Recall:    {sum(m['recall']    for m in all_metrics)/len(all_metrics):.3f}")
    print(f"  Avg F1:        {sum(m['f1']        for m in all_metrics)/len(all_metrics):.3f}")
    exact_matches = sum(1 for m in all_metrics if m["precision"] == 1.0 and m["recall"] == 1.0)
    print(f"  Exact matches: {exact_matches}/{len(all_metrics)}")

GitHub policies available:  45
Managed policies available: 30
Sampling 10 from each


GITHUB POLICIES — 10 POLICIES

[01] CodeBuildServiceRole
  Query:    Allow a build process to authenticate and manage container images in a private registry, i...
  Expected: ['ecr:BatchCheckLayerAvailability', 'ecr:BatchGetImage', 'ecr:CompleteLayerUpload', 'ecr:GetAuthorizationToken']... (14 total)
  Generated: ['ecr:BatchCheckLayerAvailability', 'ecr:BatchGetImage', 'ecr:CompleteLayerUpload', 'ecr:GetAuthorizationToken']... (11 total)
  Precision: 1.000 | Recall: 0.786 | F1: 0.880
  Missing:  ['ecr:GetDownloadUrlForLayer', 'logs:CreateLogGroup', 's3:GetObjectVersion']

[02] CodePipelineServiceRole
  Query:    Allow a deployment pipeline to initiate and monitor build processes, manage and update con...
  Expected: ['codebuild:BatchGetBuilds', 'codebuild:StartBuild', 'ecs:DescribeServices', 'ecs:DescribeTaskDefinition']... (13 total)
  Generated: ['codebuild:BatchGetBuilds', 'codebuild:BatchGetProjec

In [12]:
import json
import os

gt_path = os.path.join(DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json")
with open(gt_path) as f:
    ground_truth = json.load(f)

gt_db_path = os.path.join(DRIVE_BASE, "data/processed/ground_truth_actions.json")
with open(gt_db_path) as f:
    gt_database = json.load(f)

# Normalize valid actions — strip [permission only] and similar suffixes
valid_actions_normalized = {
    a.lower().split(' [')[0].strip()
    for a in gt_database["valid_actions"]
}

print(f"Scraped actions (normalized): {len(valid_actions_normalized)}")

uncovered    = {}
service_gaps = {}

for entry in ground_truth:
    policy_name = entry["policy_name"]
    source      = entry["source"]

    for action in entry["actions"]:
        if not isinstance(action, str) or ":" not in action:
            continue
        if "*" in action:
            continue

        # Normalize the ground truth action too
        action_normalized = action.lower().split(' [')[0].strip()

        if action_normalized not in valid_actions_normalized:
            if action not in uncovered:
                uncovered[action] = []
            uncovered[action].append(f"{policy_name} ({source})")

            prefix = action.split(":")[0].lower()
            if prefix not in service_gaps:
                service_gaps[prefix] = set()
            service_gaps[prefix].add(action)

print(f"Truly uncovered actions: {len(uncovered)}")
print(f"Services with genuine gaps: {sorted(service_gaps.keys())}")
print()
for svc in sorted(service_gaps.keys()):
    actions = sorted(service_gaps[svc])
    print(f"  {svc} ({len(actions)} actions):")
    for a in actions:
        print(f"    {a} → {uncovered[a]}")

Scraped actions (normalized): 4483
Truly uncovered actions: 3
Services with genuine gaps: ['apigateway', 'apigatewayv2']

  apigateway (1 actions):
    apigateway:GetTags → ['ApiGatewayInspectorProxyPolicy (github)']
  apigatewayv2 (2 actions):
    apigatewayv2:GET → ['ApiGatewayInspectorProxyPolicy (github)']
    apigatewayv2:GetTags → ['ApiGatewayInspectorProxyPolicy (github)']
